# AI Agent 기반 농업 정보 검색 실습 답안

외부 Open API는 쓰지 않습니다. 같은 폴더의 **농업기술길잡이 PDF 7종**과 **압축 CSV 2종**을 로컬 Tool로 연결하고, 최신 이슈만 **API 키가 필요 없는 웹 검색**으로 보강합니다.

```text
                 AI 농업 Agent
                      │
       ┌──────────────┼──────────────┐
       ↓              ↓              ↓
   농업기술 RAG    농업 CSV        웹 검색
       │              │              │
       ↓              ↓              ↓
   PDF 자료      실제 데이터     최신 정보
       └──────────────┼──────────────┘
                      ↓
                    LLM
                      ↓
              농업 정보 답변
```

핵심은 세 갈래를 매번 전부 치는 것이 아닙니다. 질문 의도에 맞는 Tool만 호출하고, Tool이 돌려준 근거만으로 답합니다.


## STEP 0. 환경 준비

`C:\env\.env`에서 **OPENAI_API_KEY만** 로드합니다. 키 값은 출력하지 않습니다.

- 기상청 키, 농업기술 데이터 플랫폼 키, Tavily/Serper/Google/Bing 검색 키는 사용하지 않습니다.
- LLM: `ChatOpenAI(model="gpt-4o-mini", temperature=0)`
- 임베딩: `OpenAIEmbeddings(model="text-embedding-3-small")`
- Agent: `from langchain.agents import create_agent`
- 압축 해제: 같은 폴더 `data/` (`노지_2024/`, `스마트팜_2024/`)
- 벡터스토어 persist: `C:\env\agri_search_faiss` (Windows FAISS 한글 경로 문제 회피)


In [1]:
import os
import re
import time
import warnings
import zipfile
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path(
    r"c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습-수업중 실습"
    r"\04_AI Agent 기반 농업 정보 검색 서비스 개발 실습"
)
PDF_DIR = DATA_DIR / "pdf"
EXTRACT_DIR = DATA_DIR / "data"
FIELD_ROOT = EXTRACT_DIR / "노지_2024"
SMART_ROOT = EXTRACT_DIR / "스마트팜_2024" / "2024"
VS_DIR = Path(r"C:\env\agri_search_faiss")
ENV_PATH = r"C:\env\.env"

load_dotenv(ENV_PATH)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(f"OPENAI_API_KEY를 {ENV_PATH}에서 찾을 수 없습니다.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", chunk_size=40)

PDF_CROPS = ("토마토", "고추", "수박", "참외", "고구마", "딸기", "사과")
FIELD_CROPS = ("마늘", "배추", "콩", "밀", "고추", "양파", "사과", "포도", "옥수수")
SMART_CROPS = (
    "가지",
    "국화",
    "딸기",
    "방울토마토",
    "수박",
    "오이",
    "완숙토마토",
    "참외",
    "파프리카",
)
TOMATO_ALIASES = ("완숙토마토", "방울토마토")

print("OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("LLM: gpt-4o-mini / Embedding: text-embedding-3-small")
print("PDF 작물:", ", ".join(PDF_CROPS))
print("노지 CSV 작물:", ", ".join(FIELD_CROPS))
print("스마트팜 CSV 작물:", ", ".join(SMART_CROPS))


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_16656\286897725.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)
LLM: gpt-4o-mini / Embedding: text-embedding-3-small
PDF 작물: 토마토, 고추, 수박, 참외, 고구마, 딸기, 사과
노지 CSV 작물: 마늘, 배추, 콩, 밀, 고추, 양파, 사과, 포도, 옥수수
스마트팜 CSV 작물: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카


## STEP 1. 자료 해제와 목록 파악

`노지_2024.zip`, `스마트팜_2024.zip`을 같은 폴더 `data/`에 풉니다. ZIP 엔트리 한글 이름이 깨지면 CP949로 복원합니다.

환경 CSV 2개는 **head만** 확인하고 전 행을 display하지 않습니다. 컬럼 목록과 샘플 1~2행만 보여 줍니다.


In [2]:
def find_zip(keyword: str) -> Path:
    hits = [p for p in DATA_DIR.glob("*.zip") if keyword in p.name]
    if not hits:
        raise FileNotFoundError(f"'{keyword}'이 들어간 zip을 같은 폴더에서 찾지 못했습니다.")
    return hits[0]


def decode_zip_name(info: zipfile.ZipInfo) -> str:
    name = info.filename
    if info.flag_bits & 0x800:
        return name
    try:
        return name.encode("cp437").decode("cp949")
    except Exception:
        return name


def extract_zip_kr(zip_path: Path, dest: Path) -> list[str]:
    dest.mkdir(parents=True, exist_ok=True)
    names = []
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = decode_zip_name(info)
            target = dest / name
            if info.is_dir() or name.endswith("/"):
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists() or target.stat().st_size != info.file_size:
                with zf.open(info) as src, open(target, "wb") as dst:
                    dst.write(src.read())
            names.append(name)
    return names


FIELD_ZIP = find_zip("노지")
SMART_ZIP = find_zip("스마트팜")
field_names = extract_zip_kr(FIELD_ZIP, EXTRACT_DIR)
smart_names = extract_zip_kr(SMART_ZIP, EXTRACT_DIR / "스마트팜_2024")

print("노지 ZIP:", FIELD_ZIP.name, f"({len(field_names)}개 파일)")
print("스마트팜 ZIP:", SMART_ZIP.name, f"({len(smart_names)}개 파일)")
print("노지 루트:", FIELD_ROOT.exists(), FIELD_ROOT)
print("스마트팜 루트:", SMART_ROOT.exists(), SMART_ROOT)


노지 ZIP: 노지_2024.zip (10개 파일)
스마트팜 ZIP: 스마트팜_2024.zip (13개 파일)
노지 루트: True c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습-수업중 실습\04_AI Agent 기반 농업 정보 검색 서비스 개발 실습\data\노지_2024
스마트팜 루트: True c:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습-수업중 실습\04_AI Agent 기반 농업 정보 검색 서비스 개발 실습\data\스마트팜_2024\2024


In [3]:
def read_csv_kr(path: Path, **kwargs) -> pd.DataFrame:
    last = None
    for enc in ("cp949", "euc-kr", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as exc:
            last = exc
    raise last


def detect_csv_encoding(path: Path) -> str:
    for enc in ("cp949", "euc-kr", "utf-8-sig", "utf-8"):
        try:
            pd.read_csv(path, encoding=enc, nrows=2)
            return enc
        except Exception:
            continue
    return "cp949"


def find_col(df: pd.DataFrame, candidates: list[str]):
    exact = {str(c).strip(): c for c in df.columns}
    for name in candidates:
        if name in exact:
            return exact[name]
    compact = {str(c).replace(" ", ""): c for c in df.columns}
    for name in candidates:
        if name.replace(" ", "") in compact:
            return compact[name.replace(" ", "")]
    return None


def crop_from_filename(name: str) -> str | None:
    for keyword in ("토마토", "고추", "수박", "참외", "고구마", "딸기", "사과"):
        if keyword in name:
            return keyword
    return None


def count_csv_rows(path: Path) -> int:
    enc = detect_csv_encoding(path)
    n = 0
    for chunk in pd.read_csv(path, encoding=enc, usecols=[0], chunksize=200_000):
        n += len(chunk)
    return n


def period_hint(df: pd.DataFrame) -> str:
    date_col = find_col(
        df,
        ["조사일", "조사일자", "출하일자", "측정시간", "정식일", "정식일자", "연도"],
    )
    if date_col is None:
        return ""
    series = df[date_col].dropna().astype(str)
    if series.empty:
        return ""
    if date_col == "연도":
        years = sorted(set(series.tolist()))
        return ", ".join(years)
    parsed = pd.to_datetime(series, errors="coerce").dropna()
    if parsed.empty:
        return series.iloc[0][:10]
    return f"{parsed.min().date()} ~ {parsed.max().date()}"


def crop_hint_from_df(df: pd.DataFrame, filename: str) -> str:
    crop_col = find_col(df, ["품목", "작목"])
    if crop_col is not None:
        values = (
            df[crop_col]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )
        if values:
            return ", ".join(sorted(set(values))[:12])
    mapped = crop_from_filename(filename)
    for crop in FIELD_CROPS + SMART_CROPS:
        if crop in filename:
            return crop
    return mapped or ""


inventory_rows = []


def add_inventory(구분: str, path: Path, is_excel: bool = False, env_head_only: bool = False):
    if is_excel:
        df = pd.read_excel(path)
        n = len(df)
        cols = list(df.columns)
        sample = df.head(2)
        period = period_hint(sample)
        crops = crop_hint_from_df(sample, path.name)
    else:
        sample = read_csv_kr(path, nrows=2)
        cols = list(sample.columns)
        period = period_hint(sample)
        crops = crop_hint_from_df(sample, path.name)
        n = count_csv_rows(path)
    inventory_rows.append(
        {
            "구분": 구분,
            "파일": str(path.relative_to(EXTRACT_DIR)).replace("\\", "/"),
            "행 수": n,
            "주요 컬럼": ", ".join(str(c) for c in cols[:12]),
            "작물": crops,
            "기간 힌트": period or "2024",
        }
    )
    return sample, cols, n


print("[노지] 농가정보·생육기본")
farm_path = FIELD_ROOT / "농가정보_2024" / "공개용_2024_농가정보.csv"
farm_sample, farm_cols, farm_n = add_inventory("노지 농가정보", farm_path)
print(f"  {farm_path.name}: {farm_n}행, 컬럼 {farm_cols}")

growth_dir = FIELD_ROOT / "생육기본_2024"
for path in sorted(growth_dir.glob("*.csv")):
    sample, cols, n = add_inventory("노지 생육기본", path)
    print(f"  {path.name}: {n}행 / 인코딩 추정 후 로드 / 컬럼 {cols}")

print("\\n[스마트팜] 재배정보·생산·생육")
cult_path = SMART_ROOT / "재배정보_2024" / "재배정보_2024.xlsx"
cult_sample, cult_cols, cult_n = add_inventory("스마트팜 재배정보", cult_path, is_excel=True)
print(f"  {cult_path.name}: {cult_n}행, 컬럼 {cult_cols}")

prod_path = SMART_ROOT / "생산_2024" / "생산_2024.csv"
prod_sample, prod_cols, prod_n = add_inventory("스마트팜 생산", prod_path)
print(f"  {prod_path.name}: {prod_n}행, 컬럼 {prod_cols}")

sf_growth_dir = SMART_ROOT / "생육_2024"
for path in sorted(sf_growth_dir.glob("*.csv")):
    sample, cols, n = add_inventory("스마트팜 생육", path)
    print(f"  {path.name}: {n}행, 컬럼 {cols}")

print("\\n[스마트팜] 환경 CSV — 전 행 display 금지, 행 수만 세고 head만 확인")
env_dir = SMART_ROOT / "환경_2024"
env_samples = {}
for path in sorted(env_dir.glob("*.csv")):
    sample, cols, n = add_inventory("스마트팜 환경", path, env_head_only=True)
    env_samples[path.name] = sample
    print(f"  {path.name}: {n}행 (head만 표시), 컬럼 {cols}")

inv_df = pd.DataFrame(inventory_rows)
display(inv_df)


[노지] 농가정보·생육기본
  공개용_2024_농가정보.csv: 312행, 컬럼 ['연도', '지역(도)', '시군', '농가명', '작목', '품종', '포장면적', '주간거리', '조간거리', '파종일자', '정식일자', '수확일자', '총수확량', '비고']
  생육기본_고추_2024.csv: 5252행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '개체번호', '초장', '착과수', '수확과수', '비고']
  생육기본_마늘_2024.csv: 13809행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '조사구역', '개체번호', '엽수', '초장', '구직경', '생구무게', '비고']
  생육기본_밀_2024.csv: 4235행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '조사구역', '초장', '수수', '천립중', '출수기', '성숙기', '비고']
  생육기본_배추_2024.csv: 2733행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '개체번호', '초장(엽장)', '엽수', '구고', '구폭', '주중', '비고']
  생육기본_사과_2024.csv: 8089행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '개체번호', '수고', '수폭', '주간굵기', '발아기', '개화시', '만개기', '수확기', '주지번호', '착과수', '엽수', '신초길이', '과일번호', '과중', '비고']
  생육기본_양파_2024.csv: 9223행 / 인코딩 추정 후 로드 / 컬럼 ['시도', '시군구', '품목', '농가명', '조사일', '조사구역', '개체번호', '엽수', '초장', '구직경', '생구무게', '비고']
  생육기본_옥수수_2024.csv: 945행 / 인코딩 추정 후 로드 / 컬럼 

,구분,파일,행 수,주요 컬럼,작물,기간 힌트
0,노지 농가정보,노지_2024/농가정보_2024/공개용_2024_농가정보.csv,312,"연도, 지역(도), 시군, 농가명, 작목, 품종, 포장면적, 주간거리, 조간거리, 파종일자, 정식일자, 수확일자",고추,2024-04-29 ~ 2024-05-01
1,노지 생육기본,노지_2024/생육기본_2024/생육기본_고추_2024.csv,5252,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 초장, 착과수, 수확과수, 비고",고추,2024-05-17 ~ 2024-05-17
2,노지 생육기본,노지_2024/생육기본_2024/생육기본_마늘_2024.csv,13809,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 개체번호, 엽수, 초장, 구직경, 생구무게, 비고",마늘,2024-04-02 ~ 2024-04-02
3,노지 생육기본,노지_2024/생육기본_2024/생육기본_밀_2024.csv,4235,"시도, 시군구, 품목, 농가명, 조사일, 조사구역, 초장, 수수, 천립중, 출수기, 성숙기, 비고",밀,2024-02-06 ~ 2024-02-06
4,노지 생육기본,노지_2024/생육기본_2024/생육기본_배추_2024.csv,2733,"시도, 시군구, 품목, 농가명, 조사일, 개체번호, 초장(엽장), 엽수, 구고, 구폭, 주중, 비고",배추,2024-09-13 ~ 2024-09-13
...,...,...,...,...,...,...
18,스마트팜 생육,스마트팜_2024/2024/생육_2024/생육_완숙토마토_2024.csv,55388,"도, 시군, 품목, 작기, 농가명, 조사일자, 개체번호, 줄기번호, 초장, 생장길이, 엽수, 엽장",완숙토마토,2024-04-17 ~ 2024-04-17
19,스마트팜 생육,스마트팜_2024/2024/생육_2024/생육_참외_2024.csv,4830,"도, 시군, 품목, 작기, 농가명, 조사일자, 개체번호, 초장, 엽장, 엽폭, 줄기굵기, 비고",참외,2025-01-07 ~ 2025-01-07
20,스마트팜 생육,스마트팜_2024/2024/생육_2024/생육_파프리카_2024.csv,20921,"도, 시군, 품목, 작기, 농가명, 조사일자, 개체번호, 줄기번호, 초장, 생장길이, 엽수, 엽장",파프리카,2024-03-19 ~ 2024-03-19
21,스마트팜 환경,"스마트팜_2024/2024/환경_2024/환경_2024(가지,국화,수박,오이,참외).csv",316766,"도, 시군, 품목, 작기, 농가명, 측정시간, 온도_외부, 풍향_외부, 풍속_외부, 일사량_외부, 누적일사량_외부, 강우감지",가지,2024-08-02 ~ 2024-08-02


In [4]:
print("노지 농가정보 샘플 1~2행")
display(farm_sample)

print("노지 고추 생육 샘플 1~2행")
pepper_growth = FIELD_ROOT / "생육기본_2024" / "생육기본_고추_2024.csv"
display(read_csv_kr(pepper_growth, nrows=2))

print("노지 사과 생육은 utf-8-sig일 수 있음 — 샘플")
apple_growth = FIELD_ROOT / "생육기본_2024" / "생육기본_사과_2024.csv"
print("  인코딩:", detect_csv_encoding(apple_growth))
display(read_csv_kr(apple_growth, nrows=2))

print("스마트팜 재배정보 샘플")
display(cult_sample.head(2))

print("스마트팜 생산 샘플")
display(prod_sample)

print("스마트팜 수박 생육 — '도명' 컬럼 확인")
wm_growth = SMART_ROOT / "생육_2024" / "생육_수박_2024.csv"
wm_sample = read_csv_kr(wm_growth, nrows=2)
print("  컬럼:", list(wm_sample.columns))
display(wm_sample)

print("환경 CSV head만 (잔존 CO2 / 잔존CO2 불일치 확인)")
for name, sample in env_samples.items():
    print(f"\\n[{name}]")
    print("  컬럼:", list(sample.columns))
    display(sample.head(2))


노지 농가정보 샘플 1~2행


,연도,지역(도),시군,농가명,작목,품종,포장면적,주간거리,조간거리,파종일자,정식일자,수확일자,총수확량,비고
0,2024,경기,안성,1,고추,뚝심칼탄,660,0.26,NaN,NaN,2024-04-29,2024-10-02,336,NaN
1,2024,경기,화성,2,고추,칼탄맥스,1155,0.45,NaN,NaN,2024-05-01,2024-09-03,139,NaN


노지 고추 생육 샘플 1~2행


,시도,시군구,품목,농가명,조사일,개체번호,초장,착과수,수확과수,비고
0,경기,안성,고추,1,2024-05-17,1,30.5,0,0,NaN
1,경기,안성,고추,1,2024-05-17,2,37.1,0,0,NaN


노지 사과 생육은 utf-8-sig일 수 있음 — 샘플
  인코딩: utf-8-sig


...

,도명,시군,품목,작기,농가명,조사일자,개체번호,마디수,엽수,줄기굵기,엽장,엽폭,엽병장,절간장,착과절위,착과율,수확율,수확일수,과장,과폭,과피두께,비고
0,충북,음성,수박,1,1,2024-05-02,1,10,10,6.8,16.9,15.9,10.7,10.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,충북,음성,수박,1,1,2024-05-02,2,13,13,6.7,21.3,22.5,15.1,11.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


환경 CSV head만 (잔존 CO2 / 잔존CO2 불일치 확인)
\n[환경_2024(가지,국화,수박,오이,참외).csv]
  컬럼: ['도', '시군', '품목', '작기', '농가명', '측정시간', '온도_외부', '풍향_외부', '풍속_외부', '일사량_외부', '누적일사량_외부', '강우감지', '온도_내부', '상대습도_내부', '잔존 CO2', '토양온도']


,도,시군,품목,작기,농가명,측정시간,온도_외부,풍향_외부,풍속_외부,일사량_외부,누적일사량_외부,강우감지,온도_내부,상대습도_내부,잔존 CO2,토양온도
0,경기,안성,가지,1,1,2024-08-02 0:00,27.8,156,0.6,0,1807,0,27.9,91.7,400.9,28.5
1,경기,안성,가지,1,1,2024-08-02 1:00,27.7,118,0.2,0,1807,0,27.7,93.1,412.3,28.3


\n[환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv]
  컬럼: ['도', '시군', '품목', '작기', '농가명', '측정시간', '온도_외부', '풍향_외부', '풍속_외부', '일사량_외부', '누적일사량_외부', '강우감지', '온도_내부', '상대습도_내부', '잔존CO2', '토양온도', 'Unnamed: 16']


,도,시군,품목,작기,농가명,측정시간,온도_외부,풍향_외부,풍속_외부,일사량_외부,누적일사량_외부,강우감지,온도_내부,상대습도_내부,잔존CO2,토양온도,Unnamed: 16
0,충북,산청,딸기,1,1,2024-10-19 9:02,NaN,NaN,NaN,0,NaN,NaN,20.1,91.7,NaN,NaN,NaN
1,충북,산청,딸기,1,1,2024-10-19 10:02,NaN,NaN,NaN,0,NaN,NaN,20.9,89.7,NaN,NaN,NaN


## STEP 2. 농업기술 RAG Tool (PDF)

`pdf/`의 농업기술길잡이 **7종**을 모두 로드합니다. 청크 메타데이터는 `crop`, `source`, `page`입니다.

- 청킹: `RecursiveCharacterTextSplitter` (chunk_size 1000, overlap 120)
- 벡터스토어: FAISS, persist `C:\env\agri_search_faiss` (재실행 시 재임베딩하지 않음)
- Tool: `search_crop_guide(query, crop="")` — crop이 있으면 해당 작물 청크만 검색


In [5]:
CROP_KEYWORDS = [
    ("토마토", "토마토"),
    ("고추", "고추"),
    ("수박", "수박"),
    ("참외", "참외"),
    ("고구마", "고구마"),
    ("딸기", "딸기"),
    ("사과", "사과"),
]


def crop_from_pdf_name(name: str) -> str | None:
    for keyword, crop in CROP_KEYWORDS:
        if keyword in name:
            return crop
    return None


def list_pdf_files() -> list[Path]:
    pdfs = [
        path
        for path in PDF_DIR.iterdir()
        if path.suffix.lower() == ".pdf" and crop_from_pdf_name(path.name) in PDF_CROPS
    ]
    return sorted(pdfs, key=lambda p: crop_from_pdf_name(p.name) or p.name)


def load_pdf_documents(pdf_paths: list[Path]) -> list[Document]:
    docs: list[Document] = []
    for path in pdf_paths:
        crop = crop_from_pdf_name(path.name)
        reader = PdfReader(str(path))
        kept = 0
        for idx, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()
            text = re.sub(r"[ \t]+", " ", text)
            text = re.sub(r"\n{3,}", "\n\n", text)
            if len(text) < 80:
                continue
            docs.append(
                Document(
                    page_content=text,
                    metadata={"crop": crop, "source": path.name, "page": idx},
                )
            )
            kept += 1
        print(f"  {crop}: {path.name} → {len(reader.pages)}쪽 중 {kept}쪽 사용")
    return docs


pdf_files = list_pdf_files()
print("인덱싱 대상 PDF:", len(pdf_files), "종")
for p in pdf_files:
    print(" ", crop_from_pdf_name(p.name), "←", p.name)
if len(pdf_files) != 7:
    raise FileNotFoundError("농업기술길잡이 PDF 7종을 pdf/ 폴더에서 찾지 못했습니다.")

index_file = VS_DIR / "index.faiss"
if index_file.exists():
    print("기존 FAISS 인덱스를 로드합니다. 재임베딩하지 않습니다.")
    vectorstore = FAISS.load_local(
        str(VS_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print("로드된 벡터 수:", vectorstore.index.ntotal)
else:
    print("PDF를 로드하고 벡터스토어를 새로 만듭니다. (최초 1회)")
    raw_docs = load_pdf_documents(pdf_files)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(raw_docs)
    print("원본 페이지 문서:", len(raw_docs), "/ 청크:", len(chunks))

    def add_batch(store, batch, start_idx: int):
        for attempt in range(8):
            try:
                if store is None:
                    return FAISS.from_documents(batch, embeddings)
                store.add_documents(batch)
                return store
            except Exception as exc:
                msg = str(exc).lower()
                if "rate" in msg or "429" in msg:
                    wait = 20 + attempt * 10
                    print(f"  rate limit at {start_idx}, {wait}초 대기 (시도 {attempt + 1}/8)")
                    time.sleep(wait)
                    continue
                raise
        raise RuntimeError("임베딩 rate limit을 여러 번 재시도했지만 실패했습니다.")

    vectorstore = None
    batch_size = 40
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        print(f"  임베딩 {start + 1}~{start + len(batch)} / {len(chunks)}")
        vectorstore = add_batch(vectorstore, batch, start + 1)
        time.sleep(1)

    VS_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(VS_DIR))
    print("FAISS persist 완료:", VS_DIR)


인덱싱 대상 PDF: 7 종
  고구마 ← 농업기술길잡이28_고구마.PDF
  고추 ← 5 고추 최종파일_단면.pdf
  딸기 ← 농업기술길잡이40_딸기.PDF
  사과 ← 농업기술길잡이5_사과재배.PDF
  수박 ← 수박_단면.pdf
  참외 ← 농업기술길잡이105_참외.PDF
  토마토 ← 29 토마토_저화질_단면.pdf
기존 FAISS 인덱스를 로드합니다. 재임베딩하지 않습니다.
로드된 벡터 수: 2160


In [6]:
def format_crop_hits(docs: list[Document]) -> str:
    if not docs:
        return "검색된 농업기술길잡이 문맥이 없습니다. 없는 내용은 만들어 내지 마세요."
    blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        excerpt = doc.page_content.strip().replace("\n", " ")
        if len(excerpt) > 700:
            excerpt = excerpt[:700] + "..."
        blocks.append(
            f"[{i}] 작물={meta.get('crop')} | 파일={meta.get('source')} | 페이지={meta.get('page')}\n{excerpt}"
        )
    return "아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.\n\n" + "\n\n".join(blocks)


@tool
def search_crop_guide(query: str, crop: str = "") -> str:
    """농촌진흥청 농업기술길잡이 PDF에서 작물 재배 지식을 검색합니다.
    가능한 작물: 토마토, 고추, 수박, 참외, 고구마, 딸기, 사과.
    정식/육묘/시비/병해충 방제/물 관리 등 재배 매뉴얼에 사용하세요.
    검색된 발췌에 없는 재배법·약제명·수치를 만들지 마세요.
    목록에 없는 작물은 다른 작물 내용으로 대체하지 말고 자료 없음을 반환합니다.
    실측 숫자나 올해 뉴스가 필요하면 이 도구 대신 CSV/웹 도구를 쓰세요.

    Args:
        query: 찾고 싶은 재배 내용 (예: 정식과 물 관리 요령).
        crop: 작물명. 토마토/고추/수박/참외/고구마/딸기/사과 중 하나. 없으면 빈 문자열.
    """
    crop = (crop or "").strip()
    if crop and crop not in PDF_CROPS:
        return (
            f"제공된 농업기술길잡이 PDF에 '{crop}' 자료가 없습니다. "
            f"가능한 작물: {', '.join(PDF_CROPS)}. "
            "다른 작물 내용을 해당 작물인 것처럼 사용하지 마세요. "
            "주어진 자료만으로는 알 수 없다고 답하세요."
        )
    try:
        search_kwargs = {"k": 5}
        if crop:
            search_kwargs["filter"] = {"crop": crop}
            search_kwargs["fetch_k"] = 40
        docs = vectorstore.similarity_search(query, **search_kwargs)
        if crop:
            docs = [d for d in docs if d.metadata.get("crop") == crop]
        return format_crop_hits(docs)
    except Exception as exc:
        return f"작물 길잡이 검색 실패: {type(exc).__name__}: {exc}"


print("작물 검색 Tool 정의 완료:", search_crop_guide.name)
sample = search_crop_guide.invoke({"query": "정식과 물 관리", "crop": "고구마"})
print("고구마 검색 미리보기:\\n")
print(sample[:900])


작물 검색 Tool 정의 완료: search_crop_guide
고구마 검색 미리보기:\n
아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.

[1] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=193
193제Ⅵ장 병해충 방제 및 생리장해 • 이병주 소각  발병된 식물체가 발견되면 소각하여야 한다. 그러나 피해 정도가 적어  회생이 가능할 경우 유기물을 시용하고 관수하면 회복될 수 있으나 수 량이 감소되므로 상황에 따라 조치한다. 감염된 식물체는 포장에 넣지  말고 소각하거나 사료로 이용한다. 묘상은 3년마다 이동하거나 상토를  교체하는 것이 좋다.  • 감염방지를 위한 묘 삽식 전처리  묘를 채취할 때는 땅이 닿는 부위에서 2～5cm 부위를 잘라 절단 부위 의 균 침입을 막는다. 자른 묘는 100개씩 다발을 지어 깨끗한 저장고나  음지에 2～3일 저장하여 두면 절단 부위의 상처가 아물어 치유(큐어링) 되고 삽식 후 발근이 촉진되어 병 저항성이 높아지는 등 적응력이 높아 진다. 또 삽식작업이 바쁠 경우 살균제인 베노밀 수화제 등에 침지하였 다가 삽식할 수 있으나 효과는 묘 아물이만 못하다<그림 6-10>.  <그림 6-10> 삽수 아물이 효과(전남농업기술원, 1999∼2000) ※ 봉산통 절토개간지 아물이


## STEP 3. 농업 CSV Tool (로컬 실측)

ZIP에서 푼 CSV/XLSX만 읽습니다. 포털·Open API를 다시 치지 않습니다.

- `query_field_csv(crop, kind, region="")` — 노지 농가정보(`farm`) · 생육기본(`growth`)
- `query_smartfarm_csv(crop, kind, region="", start_date="", end_date="")` — 재배정보·환경·생육·생산

환경 CSV는 품목(필요하면 기간·지역)으로 필터한 뒤 **평균·최소·최대·건수만** 반환합니다. 원본 수십만 행을 그대로 반환하지 않습니다.

`토마토` 요청 시 스마트팜에서는 **완숙토마토**와 **방울토마토**를 구분해 조회하고, 둘 다 조회했음을 명시합니다. 해당 작물/kind 파일이 없으면 `"자료 없음"`을 반환합니다.


In [7]:
SKIP_NUMERIC = {
    "연도",
    "연번",
    "농가명",
    "작기",
    "개체번호",
    "줄기번호",
    "화방번호",
    "조사구역",
    "조사번호",
    "과일번호",
    "주지번호",
    "결과지번호",
    "풍향_외부",
    "강우감지",
}


def resolve_smart_crops(crop: str) -> list[str]:
    crop = (crop or "").strip()
    if crop == "토마토":
        return list(TOMATO_ALIASES)
    if crop in SMART_CROPS:
        return [crop]
    return []


def region_mask(df: pd.DataFrame, region: str) -> pd.Series:
    if not region:
        return pd.Series(True, index=df.index)
    cols = []
    for name in ("지역(도)", "시도", "도", "도명", "시군", "시군구"):
        col = find_col(df, [name])
        if col is not None:
            cols.append(col)
    if not cols:
        return pd.Series(True, index=df.index)
    mask = pd.Series(False, index=df.index)
    for col in cols:
        mask = mask | df[col].astype(str).str.contains(region, na=False)
    return mask


def date_mask(df: pd.DataFrame, start_date: str, end_date: str, candidates: list[str]) -> pd.Series:
    col = find_col(df, candidates)
    if col is None or (not start_date and not end_date):
        return pd.Series(True, index=df.index)
    parsed = pd.to_datetime(df[col], errors="coerce")
    mask = parsed.notna()
    if start_date:
        mask = mask & (parsed >= pd.to_datetime(start_date))
    if end_date:
        mask = mask & (parsed <= pd.to_datetime(end_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))
    return mask


def crop_mask(df: pd.DataFrame, crop: str) -> pd.Series:
    col = find_col(df, ["품목", "작목"])
    if col is None:
        return pd.Series(True, index=df.index)
    return df[col].astype(str).str.strip() == crop


def summarize_numeric(df: pd.DataFrame, prefer: list[str] | None = None) -> str:
    lines = []
    ordered = list(prefer or []) + [c for c in df.columns if c not in (prefer or [])]
    seen = set()
    for col in ordered:
        if col in seen or col not in df.columns or col in SKIP_NUMERIC:
            continue
        seen.add(col)
        s = pd.to_numeric(df[col], errors="coerce")
        valid = int(s.notna().sum())
        if valid < 3:
            continue
        lines.append(
            f"  - {col}: 평균 {s.mean():.2f}, 최소 {s.min():.2f}, 최대 {s.max():.2f} (유효 {valid}건)"
        )
    return "\n".join(lines) if lines else "  (요약할 수치 컬럼 없음)"


def region_breakdown(df: pd.DataFrame, metric_cols: list[str], top_n: int = 8) -> str:
    region_col = find_col(df, ["시도", "지역(도)", "도", "도명"])
    if region_col is None or df.empty:
        return ""
    lines = ["  [지역별]"]
    grouped = df.groupby(df[region_col].astype(str), dropna=False)
    rows = []
    for region, part in grouped:
        item = {"지역": region, "건수": len(part)}
        for col in metric_cols:
            if col in part.columns:
                s = pd.to_numeric(part[col], errors="coerce")
                if s.notna().any():
                    item[col] = round(float(s.mean()), 2)
        rows.append(item)
    rows = sorted(rows, key=lambda x: x["건수"], reverse=True)[:top_n]
    for row in rows:
        extras = ", ".join(f"{k} 평균 {v}" for k, v in row.items() if k not in {"지역", "건수"})
        extra_txt = f", {extras}" if extras else ""
        lines.append(f"  - {row['지역']}: {row['건수']}건{extra_txt}")
    return "\n".join(lines)


def field_growth_path(crop: str) -> Path | None:
    path = FIELD_ROOT / "생육기본_2024" / f"생육기본_{crop}_2024.csv"
    return path if path.exists() else None


def smart_growth_path(crop: str) -> Path | None:
    path = SMART_ROOT / "생육_2024" / f"생육_{crop}_2024.csv"
    return path if path.exists() else None


ENV_FILES = {
    "가지": "환경_2024(가지,국화,수박,오이,참외).csv",
    "국화": "환경_2024(가지,국화,수박,오이,참외).csv",
    "수박": "환경_2024(가지,국화,수박,오이,참외).csv",
    "오이": "환경_2024(가지,국화,수박,오이,참외).csv",
    "참외": "환경_2024(가지,국화,수박,오이,참외).csv",
    "딸기": "환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv",
    "방울토마토": "환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv",
    "완숙토마토": "환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv",
    "파프리카": "환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv",
}


def summarize_environment(crop: str, region: str = "", start_date: str = "", end_date: str = "") -> str:
    fname = ENV_FILES.get(crop)
    if not fname:
        return f"자료 없음: 스마트팜 환경 CSV에 '{crop}' 품목이 없습니다."
    path = SMART_ROOT / "환경_2024" / fname
    enc = detect_csv_encoding(path)
    header = pd.read_csv(path, encoding=enc, nrows=0)
    temp_col = find_col(header, ["온도_내부"])
    hum_col = find_col(header, ["상대습도_내부"])
    co2_col = find_col(header, ["잔존CO2", "잔존 CO2"])
    soil_col = find_col(header, ["토양온도"])
    rad_col = find_col(header, ["일사량_외부"])
    metric_cols = [c for c in [temp_col, hum_col, co2_col, soil_col, rad_col] if c]
    usecols = []
    for name in ["품목", "도", "시군", "측정시간", *metric_cols]:
        col = find_col(header, [name]) if name not in header.columns else name
        if name in header.columns:
            usecols.append(name)
        elif col is not None:
            usecols.append(col)
    usecols = list(dict.fromkeys(usecols))

    stats = {c: {"n": 0, "sum": 0.0, "min": float("inf"), "max": float("-inf")} for c in metric_cols}
    n_rows = 0
    date_min = None
    date_max = None
    time_col = "측정시간" if "측정시간" in header.columns else None

    for chunk in pd.read_csv(path, encoding=enc, usecols=usecols, chunksize=120_000):
        part = chunk[crop_mask(chunk, crop)]
        part = part[region_mask(part, region)]
        part = part[date_mask(part, start_date, end_date, ["측정시간"])]
        if part.empty:
            continue
        n_rows += len(part)
        if time_col and time_col in part.columns:
            parsed = pd.to_datetime(part[time_col], errors="coerce").dropna()
            if not parsed.empty:
                vmin, vmax = parsed.min(), parsed.max()
                date_min = vmin if date_min is None else min(date_min, vmin)
                date_max = vmax if date_max is None else max(date_max, vmax)
        for col in metric_cols:
            s = pd.to_numeric(part[col], errors="coerce").dropna()
            if s.empty:
                continue
            stats[col]["n"] += int(len(s))
            stats[col]["sum"] += float(s.sum())
            stats[col]["min"] = min(stats[col]["min"], float(s.min()))
            stats[col]["max"] = max(stats[col]["max"], float(s.max()))

    if n_rows == 0:
        extra = f", 지역={region}" if region else ""
        extra += f", 기간={start_date}~{end_date}" if (start_date or end_date) else ""
        return f"자료 없음: 스마트팜 환경에서 '{crop}'{extra} 조건에 맞는 행이 없습니다."

    period = ""
    if date_min is not None:
        period = f"{date_min.date()} ~ {date_max.date()}"
    lines = [
        f"[스마트팜 환경] 품목={crop}, 필터 후 {n_rows}건"
        + (f", 기간 {period}" if period else "")
        + (f", 지역 필터={region}" if region else "")
        + f", 파일={fname}",
        "원본 행은 반환하지 않고 요약 통계만 제공합니다.",
    ]
    labels = {
        temp_col: "내부온도(℃)",
        hum_col: "내부상대습도(%)",
        co2_col: "잔존CO2",
        soil_col: "토양온도(℃)",
        rad_col: "일사량_외부",
    }
    for col in metric_cols:
        st = stats[col]
        if st["n"] == 0:
            lines.append(f"  - {labels.get(col, col)}: 유효값 없음")
            continue
        avg = st["sum"] / st["n"]
        lines.append(
            f"  - {labels.get(col, col)}: 평균 {avg:.2f}, 최소 {st['min']:.2f}, 최대 {st['max']:.2f} (유효 {st['n']}건)"
        )
    return "\n".join(lines)


def summarize_field_farm(crop: str, region: str = "") -> str:
    path = FIELD_ROOT / "농가정보_2024" / "공개용_2024_농가정보.csv"
    df = read_csv_kr(path)
    df = df[crop_mask(df, crop)]
    df = df[region_mask(df, region)]
    if df.empty:
        return f"자료 없음: 2024 노지 농가정보에 '{crop}' 작목이 없습니다."
    lines = [
        f"[노지 농가정보] 작목={crop}, {len(df)}건, 파일={path.name}",
        summarize_numeric(df, prefer=["포장면적", "총수확량", "주간거리", "조간거리"]),
    ]
    region_col = find_col(df, ["지역(도)", "시도"])
    if region_col is not None:
        lines.append(region_breakdown(df, ["포장면적", "총수확량"]))
    variety_col = find_col(df, ["품종"])
    if variety_col is not None:
        varieties = df[variety_col].dropna().astype(str).value_counts().head(8)
        lines.append("  [주요 품종] " + ", ".join(f"{k} {v}건" for k, v in varieties.items()))
    return "\n".join(lines)


def summarize_field_growth(crop: str, region: str = "") -> str:
    path = field_growth_path(crop)
    if path is None:
        return (
            f"자료 없음: 2024 노지 생육기본에 '{crop}' 파일이 없습니다. "
            f"노지 생육 작물: {', '.join(FIELD_CROPS)}."
        )
    df = read_csv_kr(path)
    df = df[crop_mask(df, crop)] if find_col(df, ["품목", "작목"]) else df
    df = df[region_mask(df, region)]
    if df.empty:
        return f"자료 없음: 노지 생육 '{crop}'에서 조건에 맞는 행이 없습니다."
    date_col = find_col(df, ["조사일", "조사일자"])
    period = period_hint(df)
    prefer = ["초장", "착과수", "수확과수", "구고", "구폭", "수고", "수폭", "엽수", "분지수", "초장(엽장)"]
    lines = [
        f"[노지 생육기본] 작물={crop}, {len(df)}건, 파일={path.name}"
        + (f", 조사기간 {period}" if period else ""),
        summarize_numeric(df, prefer=prefer),
        region_breakdown(df, [c for c in prefer if c in df.columns]),
    ]
    return "\n".join([x for x in lines if x])


def summarize_smart_cultivation(crop: str, region: str = "") -> str:
    path = SMART_ROOT / "재배정보_2024" / "재배정보_2024.xlsx"
    df = pd.read_excel(path)
    df = df[crop_mask(df, crop)]
    df = df[region_mask(df, region)]
    if df.empty:
        return f"자료 없음: 스마트팜 재배정보에 '{crop}' 품목이 없습니다."
    lines = [
        f"[스마트팜 재배정보] 품목={crop}, {len(df)}건, 파일={path.name}",
        summarize_numeric(df, prefer=["식부면적", "전체면적", "재식밀도"]),
        region_breakdown(df, ["식부면적", "전체면적"]),
    ]
    for col in ["온실종류", "온실유형", "품종"]:
        if col in df.columns:
            vc = df[col].dropna().astype(str).value_counts().head(6)
            lines.append(f"  [{col}] " + ", ".join(f"{k} {v}건" for k, v in vc.items()))
    if "정식일" in df.columns:
        lines.append(f"  정식일 범위: {period_hint(df)}")
    return "\n".join([x for x in lines if x])


def summarize_smart_growth(crop: str, region: str = "", start_date: str = "", end_date: str = "") -> str:
    path = smart_growth_path(crop)
    if path is None:
        return (
            f"자료 없음: 스마트팜 생육에 '{crop}' 파일이 없습니다. "
            f"스마트팜 생육 작물: {', '.join(SMART_CROPS)}."
        )
    df = read_csv_kr(path)
    df = df[crop_mask(df, crop)] if find_col(df, ["품목"]) else df
    df = df[region_mask(df, region)]
    df = df[date_mask(df, start_date, end_date, ["조사일자", "조사일"])]
    if df.empty:
        return f"자료 없음: 스마트팜 생육 '{crop}'에서 조건에 맞는 행이 없습니다."
    prefer = [
        "초장",
        "엽수",
        "엽장",
        "엽폭",
        "줄기굵기",
        "화방별착과수",
        "관부직경",
        "생장길이",
        "착과율",
    ]
    lines = [
        f"[스마트팜 생육] 품목={crop}, {len(df)}건, 파일={path.name}"
        + (f", 조사기간 {period_hint(df)}" if period_hint(df) else ""),
        summarize_numeric(df, prefer=prefer),
        region_breakdown(df, [c for c in prefer if c in df.columns]),
    ]
    return "\n".join([x for x in lines if x])


def summarize_smart_production(crop: str, region: str = "", start_date: str = "", end_date: str = "") -> str:
    path = SMART_ROOT / "생산_2024" / "생산_2024.csv"
    df = read_csv_kr(path)
    df = df[crop_mask(df, crop)]
    df = df[region_mask(df, region)]
    df = df[date_mask(df, start_date, end_date, ["출하일자"])]
    if df.empty:
        return f"자료 없음: 스마트팜 생산(출하)에 '{crop}' 품목이 없습니다."
    lines = [
        f"[스마트팜 생산] 품목={crop}, {len(df)}건, 파일={path.name}"
        + (f", 출하기간 {period_hint(df)}" if period_hint(df) else ""),
        summarize_numeric(df, prefer=["총출하량", "판매금액"]),
        region_breakdown(df, ["총출하량", "판매금액"]),
    ]
    return "\n".join([x for x in lines if x])


@tool
def query_field_csv(crop: str, kind: str, region: str = "") -> str:
    """2024 노지 현장 농가정보·생육기본 CSV를 요약합니다.
    kind='farm'은 농가정보(포장면적, 수확량 등), kind='growth'는 생육기본(초장, 착과수 등)입니다.
    노지 작물: 마늘, 배추, 콩, 밀, 고추, 양파, 사과, 포도, 옥수수.
    딸기·토마토·수박·참외·벼는 노지 CSV에 없습니다. 자료 없음을 반환하며 다른 작물 숫자로 대체하지 마세요.
    원본 CSV 전체를 반환하지 않고 건수·지역·평균 등 요약만 줍니다.

    Args:
        crop: 작물명 (예: 고추, 사과).
        kind: 'farm' 또는 'growth'.
        region: 선택. 도/시군 부분 문자열 (예: 전북, 완주).
    """
    crop = (crop or "").strip()
    kind = (kind or "").strip().lower()
    if kind not in {"farm", "growth"}:
        return "kind는 'farm' 또는 'growth'만 가능합니다."
    if kind == "farm":
        return summarize_field_farm(crop, region)
    return summarize_field_growth(crop, region)


@tool
def query_smartfarm_csv(
    crop: str,
    kind: str,
    region: str = "",
    start_date: str = "",
    end_date: str = "",
) -> str:
    """2024 스마트팜 재배정보·환경·생육·생산 CSV를 요약합니다.
    kind: cultivation(재배정보) | environment(환경) | growth(생육) | production(출하).
    스마트팜 품목: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카.
    crop='토마토'이면 완숙토마토와 방울토마토를 구분해 조회하고 둘 다 조회했음을 명시합니다.
    사과·고추·벼 등 목록에 없는 작물은 자료 없음을 반환하며 다른 작물 숫자로 대체하지 마세요.
    환경은 품목으로 필터한 뒤 내부온도·습도·CO2 등의 평균·최소·최대·건수만 반환합니다. 원본 수십만 행을 반환하지 마세요.

    Args:
        crop: 작물명. 토마토는 완숙/방울로 확장됩니다.
        kind: cultivation | environment | growth | production
        region: 선택. 도/시군 부분 문자열.
        start_date: 선택. YYYY-MM-DD.
        end_date: 선택. YYYY-MM-DD.
    """
    crop = (crop or "").strip()
    kind = (kind or "").strip().lower()
    if kind not in {"cultivation", "environment", "growth", "production"}:
        return "kind는 cultivation, environment, growth, production 중 하나여야 합니다."
    targets = resolve_smart_crops(crop)
    if not targets:
        return (
            f"자료 없음: 스마트팜 CSV에 '{crop}' 품목이 없습니다. "
            f"가능한 품목: {', '.join(SMART_CROPS)}. "
            "다른 작물 숫자로 대체하지 마세요."
        )
    blocks = []
    if crop == "토마토":
        blocks.append("요청 작물 '토마토' → 스마트팜에서는 완숙토마토와 방울토마토를 구분해 조회합니다.")
    fn = {
        "cultivation": lambda c: summarize_smart_cultivation(c, region),
        "environment": lambda c: summarize_environment(c, region, start_date, end_date),
        "growth": lambda c: summarize_smart_growth(c, region, start_date, end_date),
        "production": lambda c: summarize_smart_production(c, region, start_date, end_date),
    }[kind]
    for item in targets:
        blocks.append(fn(item))
    return "\n\n".join(blocks)


print("CSV Tool 정의 완료:", query_field_csv.name, query_smartfarm_csv.name)
print("\\n--- 노지 고추 생육 미리보기 ---")
print(query_field_csv.invoke({"crop": "고추", "kind": "growth"})[:800])
print("\\n--- 노지 딸기(자료 없음) ---")
print(query_field_csv.invoke({"crop": "딸기", "kind": "growth"}))
print("\\n--- 스마트팜 사과(자료 없음) ---")
print(query_smartfarm_csv.invoke({"crop": "사과", "kind": "environment"}))


CSV Tool 정의 완료: query_field_csv query_smartfarm_csv
\n--- 노지 고추 생육 미리보기 ---
[노지 생육기본] 작물=고추, 5252건, 파일=생육기본_고추_2024.csv, 조사기간 2024-05-17 ~ 2024-10-14
  - 초장: 평균 88.54, 최소 17.00, 최대 179.00 (유효 5089건)
  - 착과수: 평균 37.10, 최소 0.00, 최대 285.00 (유효 4705건)
  - 수확과수: 평균 6.88, 최소 0.00, 최대 60.00 (유효 3395건)
  [지역별]
  - 경북: 3100건, 초장 평균 80.7, 착과수 평균 34.02, 수확과수 평균 5.57
  - 경기: 1000건, 초장 평균 87.43, 착과수 평균 31.82, 수확과수 평균 5.6
  - 전남: 612건, 초장 평균 112.14, 착과수 평균 59.98, 수확과수 평균 14.8
  - 충남: 540건, 초장 평균 106.69, 착과수 평균 36.84, 수확과수 평균 9.36
\n--- 노지 딸기(자료 없음) ---
자료 없음: 2024 노지 생육기본에 '딸기' 파일이 없습니다. 노지 생육 작물: 마늘, 배추, 콩, 밀, 고추, 양파, 사과, 포도, 옥수수.
\n--- 스마트팜 사과(자료 없음) ---
자료 없음: 스마트팜 CSV에 '사과' 품목이 없습니다. 가능한 품목: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카. 다른 작물 숫자로 대체하지 마세요.


## STEP 4. 웹 검색 Tool (최신 정보)

DuckDuckGo(`ddgs`) 등 **API 키가 필요 없는** 검색만 사용합니다. Tavily, Serper, Google, Bing, 네이버, 기상청, 농업기술 데이터 플랫폼 Open API는 사용하지 않습니다.

상위 3~5건의 제목·URL·짧은 발췌를 반환합니다. 실패·0건이면 `"검색 결과 없음"`을 반환하고 뉴스를 지어내지 않습니다.


In [8]:
try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS


@tool
def search_web(query: str) -> str:
    """올해 농업 이슈·정책·시세·병해충 경보 등 최신 정보를 웹에서 검색합니다.
    재배 매뉴얼은 PDF(search_crop_guide), 실측 숫자는 CSV(query_field_csv, query_smartfarm_csv)를 쓰고
    웹 검색은 올해 이슈·정책·시세·병해충 경보 등 최신 정보에만 씁니다.
    API 키가 필요 없는 DuckDuckGo 검색만 사용합니다.
    검색 스니펫의 숫자를 CSV 실측처럼 쓰지 마세요. 결과가 없으면 뉴스를 만들지 마세요.

    Args:
        query: 검색어 (예: 2026 딸기 시세 이슈).
    """
    query = (query or "").strip()
    if not query:
        return "검색 결과 없음"
    try:
        hits = []
        with DDGS() as ddgs:
            for item in ddgs.text(query, max_results=5) or []:
                hits.append(item)
        if not hits:
            return "검색 결과 없음"
        lines = ["웹 검색 결과입니다. 제목과 URL만 최신 이슈 보강에 쓰세요. 숫자는 CSV 실측으로 대체하지 마세요."]
        for i, item in enumerate(hits[:5], start=1):
            title = item.get("title") or item.get("source") or "(제목 없음)"
            url = item.get("href") or item.get("url") or ""
            body = (item.get("body") or item.get("snippet") or "").replace("\n", " ")
            if len(body) > 180:
                body = body[:180] + "..."
            lines.append(f"[{i}] 제목: {title}\n    URL: {url}\n    발췌: {body}")
        return "\n".join(lines)
    except Exception as exc:
        return f"검색 결과 없음 (검색 실패: {type(exc).__name__})"


print("웹 검색 Tool 정의 완료:", search_web.name)
preview = search_web.invoke({"query": "2026 딸기 농업 이슈"})
print(preview[:900])


웹 검색 Tool 정의 완료: search_web
웹 검색 결과입니다. 제목과 URL만 최신 이슈 보강에 쓰세요. 숫자는 CSV 실측으로 대체하지 마세요.
[1] 제목: 2026 딸기 생산·소비·수출입 동향 : 네이버 블로그
    URL: https://blog.naver.com/gcmkty/224394972498
    발췌: 딸기 재배면적은 과거보다 감소했고, 농가 고령화와 노동력 부족이라는 부담도 커지고 있습니다. 그런데 흥미로운 점이 있습니다. 재배면적은 줄었는데 최근 생산량은 단수 증가에 힘입어 회복되는 모습을 보이고 있다는 것입니다.
[2] 제목: 농림축산식품부 CMS > 맛있는 딸기, 농업 인공지능 (AI)이 키워봅...
    URL: https://www.mafra.go.kr/bbs/home/792/578053/artclView.do
    발췌: May 31, 2026 · 농림축산식품부(장관 송미령, 이하 ‘농식품부’)는 농업 분야 차세대 농업 인공지능(AI) 모델을 발굴하기 위한 ‘ 20 26 스마트 농업 인공지능(AI) 경진대회(이하 ‘경진대회’)’를 개최한다.
[3] 제목: AI 딸기 재배모델로 농업 혁신과 미래 경쟁력 강화
    URL: https://countrynews.co.kr/ko-kr/articles/472
    발췌: Jun 1, 2026 · 농업 현장에 실질적으로 도움이 되는 AI 모델을 발굴하고, 이를 통해 농업의 지속가능성과 경쟁력을 높이는 것이 이번 대회의 궁극적 목표다. 농업 은 더 이상 전통적 방식만으로는 지속가능성을 담보하기 어려운 시대에 접어들었다.
[4] 제목: [이슈브리핑] 딸기 폐기 논란 팩트체크···냉동 딸기 수입량도 줄어 - ...
    URL: https://www.newsfarm.co.kr/news/articleView.html?idxno=100065
    발췌:


## STEP 5. Agent 구성

RAG + CSV + 웹 Tool을 장착한 Agent입니다. 질문 의도에 맞는 Tool만 호출하고, 근거가 없으면 지어내지 않습니다.


In [9]:
SYSTEM_PROMPT = """당신은 농업 정보 검색 Agent입니다.
재배 지식은 농업기술길잡이 PDF, 실측 숫자는 로컬 CSV, 최신 이슈는 웹 검색 Tool 결과만 근거로 답합니다.
없는 작물·기간·수치·뉴스를 추측하지 마세요. 근거가 없으면 '주어진 자료만으로는 알 수 없다'고 말하세요.

[도구 선택 — 반드시 지키세요]
- 정식/육묘/시비/병해충 방제/물 관리 등 재배 지식 → search_crop_guide
  길잡이 기준 요약만 물으면 CSV·웹을 호출하지 마세요.
- 2024 노지 농가·생육 숫자(초장, 착과, 수확량, 지역별 실측) → query_field_csv
  스마트팜 숫자나 웹 스니펫으로 대체하지 마세요.
- 스마트팜 온실 환경·생육·출하량 → query_smartfarm_csv
  환경은 kind='environment', 출하량은 kind='production', 생육은 kind='growth'.
  온습도와 출하량을 같이 물으면 environment와 production을 각각 호출하세요.
  노지 숫자로 대체하지 마세요.
- 올해/최근/시세/발생 소식/정책/병해충 경보 등 최신 정보 → search_web
- 종합 질문이면 필요한 Tool만 조합하세요. 무관한 Tool은 호출하지 마세요.
  예: 딸기 재배 주의사항 + 스마트팜 숫자 + 최근 이슈
  → search_crop_guide + query_smartfarm_csv(growth, environment) + search_web
  노지에 딸기가 없으므로 query_field_csv는 호출하지 마세요.
- 세 갈래를 습관적으로 전부 치지 마세요.

[자료 범위]
- PDF 작물: 토마토, 고추, 수박, 참외, 고구마, 딸기, 사과
- 노지 CSV 작물: 마늘, 배추, 콩, 밀, 고추, 양파, 사과, 포도, 옥수수
- 스마트팜 CSV 품목: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카
- PDF의 토마토와 CSV의 완숙토마토/방울토마토는 같은 과채류로 조회해도 되지만, 답변에 품목명을 분명히 적으세요.
- 벼는 PDF·CSV에 없습니다. 노지 딸기 생육은 없습니다. 스마트팜 사과는 없습니다.
- 없는 자료는 다른 작물(고추, 토마토, 노지 사과 등)로 바꿔 답하지 마세요.
  해당 Tool을 호출해 자료 없음을 확인한 뒤 없다고 답하세요.

[답변 형식]
- 숫자는 CSV Tool 결과만 사용하세요. 웹 스니펫 숫자로 대체하지 마세요.
- 재배법은 PDF 발췌만 사용하세요. PDF에 없으면 웹으로 매뉴얼을 대체하지 마세요.
- 근거를 인용하세요. PDF: 파일명·페이지 / CSV: 작물·건수·기간 / 웹: 제목·URL
- 농가가 읽을 수 있는 자연어로 답하세요. JSON 원문만 출력하고 끝내지 마세요.
- 종합 질문이면 재배 요령과 실측·최신 이슈가 어떻게 연결되는지 한 단락으로 해석하되, Tool에 없는 인과는 단정하지 마세요.
- API 키를 사용자에게 출력하지 마세요.
"""

TOOLS = [search_crop_guide, query_field_csv, query_smartfarm_csv, search_web]
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)


def message_text(msg) -> str:
    content = getattr(msg, "content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("text"):
                parts.append(str(block["text"]))
        return "\n".join(parts)
    return str(content)


def extract_tool_calls(result: dict) -> list[str]:
    names = []
    for msg in result.get("messages", []):
        for call in getattr(msg, "tool_calls", None) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name:
                names.append(name)
    return names


def extract_tool_trace(result: dict) -> list[str]:
    lines = []
    for msg in result.get("messages", []):
        for call in getattr(msg, "tool_calls", None) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", "")
            args = call.get("args") if isinstance(call, dict) else getattr(call, "args", {})
            safe_args = {k: v for k, v in (args or {}).items() if "key" not in str(k).lower()}
            lines.append(f"  → {name}({safe_args})")
        if type(msg).__name__ == "ToolMessage":
            preview = message_text(msg).replace("\n", " ")[:180]
            lines.append(f"  ← Tool 결과 미리보기: {preview}...")
    return lines


def ask_agent(question: str) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 30},
    )
    called = extract_tool_calls(result)
    final = message_text(result["messages"][-1])
    return {
        "answer": final,
        "tools": called,
        "trace": extract_tool_trace(result),
        "raw": result,
    }


print("Agent 준비 완료. Tools:", [t.name for t in TOOLS])


Agent 준비 완료. Tools: ['search_crop_guide', 'query_field_csv', 'query_smartfarm_csv', 'search_web']


## STEP 6. 실행 시나리오 (A~E 실제 실행)

같은 Agent로 다섯 질문을 실행하고, 호출된 Tool 목록과 최종 답을 남깁니다.


In [10]:
def run_scenario(label: str, question: str, expected: str) -> dict:
    print("=" * 78)
    print(f"시나리오 {label}")
    print("질문:", question)
    print("기대한 Tool:", expected)
    print("=" * 78)

    out = ask_agent(question)
    called = out["tools"]
    final = out["answer"]

    print("\\n[실제 호출 Tool]")
    print(", ".join(called) if called else "(없음)")
    print("\\n[도구 호출 과정]")
    print("\\n".join(out["trace"]) if out["trace"] else "(트레이스 없음)")
    print("\\n[최종 답변]")
    print(final)
    print()

    called_set = set(called)
    record = {
        "시나리오": label,
        "질문": question,
        "질문 의도": "",
        "기대한 Tool": expected,
        "실제 호출 Tool": ", ".join(called) if called else "(없음)",
        "PDF 근거": "Y" if "search_crop_guide" in called_set else "N",
        "CSV 근거": "Y" if ("query_field_csv" in called_set or "query_smartfarm_csv" in called_set) else "N",
        "웹 근거": "Y" if "search_web" in called_set else "N",
        "최종답변": final,
        "호출목록": called,
    }
    return record


SCENARIO_RESULTS: list[dict] = []
print("시나리오 실행 함수 준비 완료")


시나리오 실행 함수 준비 완료


In [11]:
q_a = "고구마 정식과 물 관리 요령을 농업기술길잡이 기준으로 요약해줘."
SCENARIO_RESULTS.append(
    run_scenario("A", q_a, "search_crop_guide")
)


시나리오 A
질문: 고구마 정식과 물 관리 요령을 농업기술길잡이 기준으로 요약해줘.
기대한 Tool: search_crop_guide
\n[실제 호출 Tool]
search_crop_guide, search_crop_guide
\n[도구 호출 과정]
  → search_crop_guide({'query': '정식', 'crop': '고구마'})\n  → search_crop_guide({'query': '물 관리', 'crop': '고구마'})\n  ← Tool 결과 미리보기: 아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.  [1] 작물=고구마 | 파일=농업기술길잡이28_고구마.PDF | 페이지=4 제Ⅳ장 재배법 1. 재배환경 ......................................................................................\n  ← Tool 결과 미리보기: 검색된 농업기술길잡이 문맥이 없습니다. 없는 내용은 만들어 내지 마세요....
\n[최종 답변]
고구마의 정식과 물 관리 요령은 다음과 같습니다.

### 정식 요령
1. **묘의 준비**: 고구마 묘는 크기에 따라 구분하여 심어야 하며, 큰 묘와 작은 묘를 섞어 심지 않도록 주의합니다. 이는 생육에 영향을 미쳐 수량을 감소시킬 수 있습니다.
2. **심는 깊이**: 묘는 얕게 심는 것이 덩이뿌리 형성에 좋지만, 건조한 밭에서는 다소 깊게 심는 것이 바람직합니다.
3. **잎 관리**: 묘를 심을 때 잎이 떨어지지 않도록 주의해야 하며, 잎이 떨어지면 활착과 뿌리내림이 더디고 수량이 감소할 수 있습니다.
4. **심는 위치**: 묘의 선단 잎 4~6마디부터 덩이뿌리가 될 뿌리가 나타나므로, 이 부분이 땅속에 묻히도록 하되 생장점은 땅속에 묻히지 않도록 합니다.

### 물 관리 요령
- 고구마는 토양 수분에 따라 뿌리내림이 다르므로, 토양의 수분 상태를 잘 관리해야 합니다. 특히, 사질토양에서는 지온이 빨리 높아지고 건조해지기 쉬

In [12]:
q_b = (
    "2024년 노지 고추 생육을 요약해줘. 초장이나 착과가 어느 정도인지 "
    "지역별로 알려줘."
)
SCENARIO_RESULTS.append(
    run_scenario("B", q_b, "query_field_csv")
)


시나리오 B
질문: 2024년 노지 고추 생육을 요약해줘. 초장이나 착과가 어느 정도인지 지역별로 알려줘.
기대한 Tool: query_field_csv
\n[실제 호출 Tool]
query_field_csv
\n[도구 호출 과정]
  → query_field_csv({'crop': '고추', 'kind': 'growth'})\n  ← Tool 결과 미리보기: [노지 생육기본] 작물=고추, 5252건, 파일=생육기본_고추_2024.csv, 조사기간 2024-05-17 ~ 2024-10-14   - 초장: 평균 88.54, 최소 17.00, 최대 179.00 (유효 5089건)   - 착과수: 평균 37.10, 최소 0.00, 최대 285.00 (유효 4705건)   - 수확과수...
\n[최종 답변]
2024년 노지 고추의 생육 정보를 지역별로 요약하면 다음과 같습니다:

- **전체 평균**
  - 초장: 88.54 cm (최소 17.00 cm, 최대 179.00 cm)
  - 착과수: 37.10 개 (최소 0.00 개, 최대 285.00 개)
  - 수확과수: 6.88 개 (최소 0.00 개, 최대 60.00 개)

- **지역별 생육 정보**
  - **경북**
    - 초장: 평균 80.7 cm
    - 착과수: 평균 34.02 개
    - 수확과수: 평균 5.57 개
    - 건수: 3100건
  - **경기**
    - 초장: 평균 87.43 cm
    - 착과수: 평균 31.82 개
    - 수확과수: 평균 5.6 개
    - 건수: 1000건
  - **전남**
    - 초장: 평균 112.14 cm
    - 착과수: 평균 59.98 개
    - 수확과수: 평균 14.8 개
    - 건수: 612건
  - **충남**
    - 초장: 평균 106.69 cm
    - 착과수: 평균 36.84 개
    - 수확과수: 평균 9.36 개
    - 건수: 540건

이 데이터는 2024년 5월 17일부터 10월 14일까지 조사

In [13]:
q_c = "스마트팜 완숙토마토 내부 온습도가 어떤지, 출하량도 같이 요약해줘."
SCENARIO_RESULTS.append(
    run_scenario("C", q_c, "query_smartfarm_csv (environment, production)")
)


시나리오 C
질문: 스마트팜 완숙토마토 내부 온습도가 어떤지, 출하량도 같이 요약해줘.
기대한 Tool: query_smartfarm_csv (environment, production)
\n[실제 호출 Tool]
query_smartfarm_csv, query_smartfarm_csv
\n[도구 호출 과정]
  → query_smartfarm_csv({'crop': '완숙토마토', 'kind': 'environment'})\n  → query_smartfarm_csv({'crop': '완숙토마토', 'kind': 'production'})\n  ← Tool 결과 미리보기: [스마트팜 환경] 품목=완숙토마토, 필터 후 152274건, 기간 2024-01-07 ~ 2025-07-16, 파일=환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv 원본 행은 반환하지 않고 요약 통계만 제공합니다.   - 내부온도(℃): 평균 20.33, 최소 0.00, 최대 55.80 (유효 152126건) ...\n  ← Tool 결과 미리보기: [스마트팜 생산] 품목=완숙토마토, 6531건, 파일=생산_2024.csv, 출하기간 2024-03-25 ~ 2025-08-04   - 총출하량: 평균 400.41, 최소 1.00, 최대 22000.00 (유효 6531건)   - 판매금액: 평균 1236262.06, 최소 191.00, 최대 79328000.00 (유효 ...
\n[최종 답변]
스마트팜에서 재배되는 완숙토마토의 내부 온습도와 출하량에 대한 요약은 다음과 같습니다.

### 내부 온습도
- **내부온도**: 평균 20.33℃, 최소 0.00℃, 최대 55.80℃ (유효 152126건)
- **내부상대습도**: 평균 83.36%, 최소 12.20%, 최대 100.00% (유효 147117건)

### 출하량
- **총출하량**: 평균 400.41kg, 최소 1.00kg, 최대 22000.00kg (유효 6531건)
- **판매금액**: 평균 1,236,262.06원, 최

In [14]:
q_d = (
    "딸기 재배 주의사항을 길잡이에서 찾고, "
    "스마트팜 딸기 생육·환경 숫자와 최근 딸기 관련 이슈를 함께 정리해줘."
)
SCENARIO_RESULTS.append(
    run_scenario(
        "D",
        q_d,
        "search_crop_guide, query_smartfarm_csv, search_web",
    )
)


시나리오 D
질문: 딸기 재배 주의사항을 길잡이에서 찾고, 스마트팜 딸기 생육·환경 숫자와 최근 딸기 관련 이슈를 함께 정리해줘.
기대한 Tool: search_crop_guide, query_smartfarm_csv, search_web
\n[실제 호출 Tool]
search_crop_guide, query_smartfarm_csv, query_smartfarm_csv, search_web
\n[도구 호출 과정]
  → search_crop_guide({'query': '재배 주의사항', 'crop': '딸기'})\n  → query_smartfarm_csv({'crop': '딸기', 'kind': 'growth'})\n  → query_smartfarm_csv({'crop': '딸기', 'kind': 'environment'})\n  → search_web({'query': '딸기 이슈 2024'})\n  ← Tool 결과 미리보기: 아래 발췌만 근거로 답하세요. 발췌에 없는 내용은 추측하지 마세요.  [1] 작물=딸기 | 파일=농업기술길잡이40_딸기.PDF | 페이지=8 시설 토경 재배 노지 재배 고설수경재배 다단식 입체 재배 노지 육묘 비가림 평지 포트 육묘 비가림 차근 육묘 비가림 고설 포트 육묘 •딸기 육묘 및 재배 전경 190923_딸기_내지...\n  ← Tool 결과 미리보기: [스마트팜 생육] 품목=딸기, 14100건, 파일=생육_딸기_2024.csv, 조사기간 2024-09-23 ~ 2025-06-18   - 초장: 평균 35.57, 최소 8.00, 최대 64.50 (유효 13977건)   - 엽수: 평균 12.94, 최소 2.00, 최대 56.00 (유효 13983건)   - 엽장: 평균 ...\n  ← Tool 결과 미리보기: [스마트팜 환경] 품목=딸기, 필터 후 139694건, 기간 2024-09-01 ~ 2025-06-19, 파일=환경_2024(딸기,방울토마토,완숙토마토,파프리카).csv 원본 행은 반환하지 않고 요약 통계만 제공합

In [15]:
q_e = (
    "벼 논물 수심이 적절한지, 2024년 노지 딸기 생육 초장과 "
    "스마트팜 사과 내부온도까지 알려줘."
)
SCENARIO_RESULTS.append(
    run_scenario("E", q_e, "자료 없음 명시 (다른 작물로 대체하지 않음)")
)


시나리오 E
질문: 벼 논물 수심이 적절한지, 2024년 노지 딸기 생육 초장과 스마트팜 사과 내부온도까지 알려줘.
기대한 Tool: 자료 없음 명시 (다른 작물로 대체하지 않음)
\n[실제 호출 Tool]
query_field_csv, query_smartfarm_csv
\n[도구 호출 과정]
  → query_field_csv({'crop': '딸기', 'kind': 'growth'})\n  → query_smartfarm_csv({'crop': '사과', 'kind': 'environment'})\n  ← Tool 결과 미리보기: 자료 없음: 2024 노지 생육기본에 '딸기' 파일이 없습니다. 노지 생육 작물: 마늘, 배추, 콩, 밀, 고추, 양파, 사과, 포도, 옥수수....\n  ← Tool 결과 미리보기: 자료 없음: 스마트팜 CSV에 '사과' 품목이 없습니다. 가능한 품목: 가지, 국화, 딸기, 방울토마토, 수박, 오이, 완숙토마토, 참외, 파프리카. 다른 작물 숫자로 대체하지 마세요....
\n[최종 답변]
주어진 자료만으로는 알 수 없습니다. 2024년 노지 딸기 생육 초장에 대한 정보는 없으며, 스마트팜 사과에 대한 내부온도 데이터도 없습니다.



## STEP 7. 결과 정리

시나리오 A~E의 질문 의도, 기대한 Tool, 실제 호출 Tool, 근거 여부를 표로 남깁니다. (D)의 최종 자연어 답변 전문도 함께 보존합니다.


In [16]:
INTENT = {
    "A": "재배지식(PDF만)",
    "B": "실측데이터(노지 CSV만)",
    "C": "실측데이터(스마트팜 CSV만)",
    "D": "종합(RAG+CSV+웹)",
    "E": "자료 없음 / 잘못된 라우팅 방지",
}
NOTE = {
    "A": "CSV·웹을 치지 않는 것이 바람직",
    "B": "스마트팜 토마토 숫자로 고추를 대체하면 안 됨",
    "C": "환경 원문 전체 출력 금지, 노지 고추로 대체 금지",
    "D": "PDF 파일명·페이지 + CSV 수치 + 웹 제목·URL",
    "E": "벼/노지 딸기/스마트팜 사과를 다른 작물로 대체하면 안 됨",
}

summary_rows = []
for rec in SCENARIO_RESULTS:
    label = rec["시나리오"]
    summary_rows.append(
        {
            "시나리오": label,
            "질문 의도": INTENT[label],
            "기대한 Tool": rec["기대한 Tool"],
            "실제 호출 Tool": rec["실제 호출 Tool"],
            "PDF 근거": rec["PDF 근거"],
            "CSV 근거": rec["CSV 근거"],
            "웹 근거": rec["웹 근거"],
            "비고": NOTE[label],
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,시나리오,질문 의도,기대한 Tool,실제 호출 Tool,PDF 근거,CSV 근거,웹 근거,비고
0,A,재배지식(PDF만),search_crop_guide,"search_crop_guide, search_crop_guide",Y,N,N,CSV·웹을 치지 않는 것이 바람직
1,B,실측데이터(노지 CSV만),query_field_csv,query_field_csv,N,Y,N,스마트팜 토마토 숫자로 고추를 대체하면 안 됨
2,C,실측데이터(스마트팜 CSV만),"query_smartfarm_csv (environment, production)","query_smartfarm_csv, query_smartfarm_csv",N,Y,N,"환경 원문 전체 출력 금지, 노지 고추로 대체 금지"
3,D,종합(RAG+CSV+웹),"search_crop_guide, query_smartfarm_csv, search_web","search_crop_guide, query_smartfarm_csv, query_smartfarm_csv, search_web",Y,Y,Y,PDF 파일명·페이지 + CSV 수치 + 웹 제목·URL
4,E,자료 없음 / 잘못된 라우팅 방지,자료 없음 명시 (다른 작물로 대체하지 않음),"query_field_csv, query_smartfarm_csv",N,Y,N,벼/노지 딸기/스마트팜 사과를 다른 작물로 대체하면 안 됨


In [17]:
d_rec = next(r for r in SCENARIO_RESULTS if r["시나리오"] == "D")
print("시나리오 D 호출 Tool:", d_rec["실제 호출 Tool"])
print("-" * 78)
print("시나리오 D 최종 조언 (농가가 읽을 수 있는 문장)")
print("-" * 78)
print(d_rec["최종답변"])


시나리오 D 호출 Tool: search_crop_guide, query_smartfarm_csv, query_smartfarm_csv, search_web
------------------------------------------------------------------------------
시나리오 D 최종 조언 (농가가 읽을 수 있는 문장)
------------------------------------------------------------------------------
딸기 재배에 대한 주의사항과 관련된 정보는 다음과 같습니다.

### 재배 주의사항
1. **정식 전 준비**: 정식 전에 토양 소독을 실시하고, 기비로 10a당 완숙퇴비 3~4t, 석회 고토 100~150kg을 시비해야 합니다. 질소-인산-칼륨은 10a당 총 9.6-4.9-7.4kg이 필요하며, 이 중 3.5-4.9-5.6kg을 밑거름으로 줍니다. 이랑 높이는 35cm 이상으로 설정하고 2줄 심기를 권장합니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 126)

2. **정식 시기**: 반촉성재배의 정식 시기는 10월 상중순경으로, 이 시기를 지나면 뿌리 발육이 부진해져 수량이 저하될 수 있습니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 126)

3. **보온 관리**: 보온 시기는 수확기의 생육을 조절하는 중요한 요소로, 중부 지역에서는 10월 중하순부터 휴면에 들어가 1월 중순에 완료됩니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 126)

4. **수확 요령**: 수확 시 물리적 손상을 최소화하고, 수확한 과실은 적절한 온도에서 관리해야 합니다. 기온이 높은 시간에는 과실이 손상될 수 있으므로 주의해야 합니다. (출처: 농업기술길잡이40_딸기.PDF, 페이지 150)

### 스마트팜 딸기 생육 및 환경 데이터
- **생육 데이터**:
  - 초장: 평균 35.57cm (최소 8.00cm, 최대 64.50cm)
  - 엽수: 평균 12.94 

## 체크리스트

- [x] OPENAI_API_KEY만 `C:\env\.env`에서 읽었고 키 값을 출력하지 않음
- [x] 기상청·농업기술 데이터 플랫폼·유료 검색 API를 쓰지 않음
- [x] `from langchain.agents import create_agent` 사용
- [x] PDF 7종 인덱싱, crop/source/page 메타데이터
- [x] FAISS persist로 재실행 시 재임베딩 생략
- [x] 노지 ZIP·스마트팜 ZIP을 풀어 CSV Tool이 로컬 파일만 읽음
- [x] 환경 CSV를 필터·요약하고 원문 전체를 반환하지 않음
- [x] 웹 검색이 API 키 없이 동작하고 제목·URL을 반환
- [x] 시나리오 A~E 실제 실행, 도구 호출 트레이스와 비교표 보존
- [x] 최종 답이 JSON 덤프가 아니라 자연어 조언
